In [3]:
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

print(sklearn.__version__)

1.6.1


In [4]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

fake['outcome'] = 0
true['outcome'] = 1

merged = pd.concat([fake, true], ignore_index = True)

In [5]:
df = merged.drop(['title', 'subject', 'date'], axis=1)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

display(df)

,text,outcome
0,"21st Century Wire says Ben Stein, reputable pr...",0
1,WASHINGTON (Reuters) - U.S. President Donald T...,1
2,(Reuters) - Puerto Rico Governor Ricardo Rosse...,1
3,"On Monday, Donald Trump once again embarrassed...",0
4,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",1
...,...,...
44893,,0
44894,LONDON/TOKYO (Reuters) - British Prime Ministe...,1
44895,BERLIN (Reuters) - Chancellor Angela Merkel sa...,1
44896,Jesus f*cking Christ our President* is a moron...,0


In [6]:
x = df['text']
y = df['outcome']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer


vectorization = TfidfVectorizer()
xvector_train = vectorization.fit_transform(x_train)
xvector_test = vectorization.transform(x_test)

In [8]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=1)
tree.fit(xvector_train, y_train)

predict_tree = tree.predict(xvector_test)

tree.score(xvector_test, y_test)


0.995456570155902

In [9]:
# Confusion Matrix and other metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, predict_tree))
print('\n')
print('\n')
print('\n')
print(classification_report(y_test, predict_tree))

[[5789   19]
 [  32 5385]]






              precision    recall  f1-score   support

           0       0.99      1.00      1.00      5808
           1       1.00      0.99      1.00      5417

    accuracy                           1.00     11225
   macro avg       1.00      1.00      1.00     11225
weighted avg       1.00      1.00      1.00     11225



In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=1)

rf.fit(xvector_train, y_train)
rf_pred = rf.predict(xvector_test)

rf.score(xvector_test, y_test)


0.9879732739420936

In [11]:

print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5808
           1       0.99      0.99      0.99      5417

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [12]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=1)
lr.fit(xvector_train, y_train)

lr_pred = lr.predict(xvector_test)
lr.score(xvector_test, y_test)

0.9862806236080178

In [13]:
from sklearn.metrics import classification_report
print(classification_report(y_test, lr_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      5808
           1       0.98      0.99      0.99      5417

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [14]:
from sklearn.neural_network import MLPClassifier

nn = MLPClassifier(hidden_layer_sizes=(5,2))

nn.fit(xvector_train, y_train)

MLPClassifier(hidden_layer_sizes=(5, 2))

In [17]:
nn_pred = nn.predict(xvector_test)

nn.score(xvector_test, y_test)

0.9910022271714922

In [ ]:
# from sklearn import svm

# model = svm.SVC()

# model.fit(xvector_train, y_train)

# pred  = model.predict(xvector_test)
# model.score(xvector_test, y_test)

# Try a different model

In [18]:
import joblib

# save the models including the vectorization for testing against new articles
joblib.dump(tree, 'decision_tree_model.pkl')
joblib.dump(rf, 'random_forest_model.pkl')
joblib.dump(lr, 'logistic_regression_model.pk1')
joblib.dump(nn, 'neural_network_model.pkl')

joblib.dump(vectorization, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

In [24]:
def predict_fake_news(article_text):
    # Load the saved models and vectorizer
    vectorizer = joblib.load('tfidf_vectorizer.pkl')
    dt_model = joblib.load('decision_tree_model.pkl')
    rf_model = joblib.load('random_forest_model.pkl')
    lr_model = joblib.load('logistic_regression_model.pk1')
    nn_model = joblib.load('neural_network_model.pkl')
    
    # Vectorize the input text
    text_vector = vectorizer.transform([article_text])
    
    # Get predictions from each model
    dt_pred = dt_model.predict(text_vector)[0]
    rf_pred = rf_model.predict(text_vector)[0]
    lr_pred = lr_model.predict(text_vector)[0]
    nn_pred = nn_model.predict(text_vector)[0]
    
    # Calculate confidence scores
    dt_prob = dt_model.predict_proba(text_vector)[0]
    rf_prob = rf_model.predict_proba(text_vector)[0]
    lr_prob = lr_model.predict_proba(text_vector)[0]
    nn_prob = nn_model.predict_proba(text_vector)[0]
    
    # Create result dictionary
    results = {
        'decision_tree': {'prediction': 'Real' if dt_pred == 1 else 'Fake', 
                         'confidence': max(dt_prob)},
        'random_forest': {'prediction': 'Real' if rf_pred == 1 else 'Fake',
                         'confidence': max(rf_prob)},
        'Logistic Regression':{'prediction': 'Real' if lr_pred == 1 else 'Fake',
                          'confidence': max(lr_prob)},
        'neural_network': {'prediction': 'Real' if nn_pred == 1 else 'Fake',
                          'confidence': max(nn_prob)}                  
    }
    
    # Calculate ensemble prediction (majority vote)
    predictions = [dt_pred, rf_pred, nn_pred]
    ensemble_pred = 'Real' if sum(predictions) > len(predictions)/2 else 'Fake'
    avg_confidence = (max(dt_prob) + max(rf_prob) + max(lr_prob) + max(nn_prob)) / 4
    
    results['ensemble'] = {'prediction': ensemble_pred, 'confidence': avg_confidence}
    
    return results

In [29]:
text = str(input("paste article"))
predict_fake_news(text)

{'decision_tree': {'prediction': 'Fake', 'confidence': np.float64(1.0)},
 'random_forest': {'prediction': 'Fake', 'confidence': np.float64(0.57)},
 'Logistic Regression': {'prediction': 'Fake',
  'confidence': np.float64(0.8028040800834115)},
 'neural_network': {'prediction': 'Fake',
  'confidence': np.float64(0.9997804243956671)},
 'ensemble': {'prediction': 'Fake',
  'confidence': np.float64(0.8431461261197696)}}